In [2]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## DATA PREPARATION

In [3]:
import time  # not in the given import cell; needed for time.sleep

GROUP_NUMBER = "6"        
from pathlib import Path
TICKER_FILE = Path.home() / "Desktop" / "Tickers.csv"
assert TICKER_FILE.exists(), f"Can't find {TICKER_FILE}. Check the file name/location"

PURCHASE_DATE = pd.Timestamp("2026-11-27")
# Competition holding window (used to find earnings reports during the competition)
HOLD_START = pd.Timestamp("2026-11-27 16:00", tz="America/New_York")
HOLD_END   = pd.Timestamp("2026-12-04 16:00", tz="America/New_York")

# Price history to download: must start before the volume window and cover a year of signals
HISTORY_START = pd.Timestamp("2024-10-01")

# Volume rule
VOLUME_START = pd.Timestamp("2025-10-01")
VOLUME_END   = pd.Timestamp("2026-09-30")
MIN_AVG_VOLUME = 5000
MIN_TRADING_DAYS_IN_MONTH = 18

# Client mandates
CLIENT_BUDGETS = {"A": 1_000_000, "B": 750_000}       # CAD
CLIENT_N_STOCKS = {"A": 12, "B": 20}                   # our chosen portfolio sizes
MIN_STOCKS, MAX_STOCKS = 10, 20
MAX_STOCK_WEIGHT = 0.15
MAX_SECTOR_WEIGHT = 0.40

# Transaction fee: the smaller of US$2.15 and US$0.001 per share, for every stock
MAX_FEE_USD = 2.15
FEE_PER_SHARE_USD = 0.001
SAFETY_BUFFER_CAD = 0.01     # leave one cent unspent so rounding can never exceed the budget

# Eligibility (yfinance codes)
ALLOWED_QUOTE_TYPES = ["EQUITY"]
ALLOWED_CURRENCIES  = ["CAD", "USD"]
US_EXCHANGES = ["NMS", "NGM", "NCM", "NYQ", "ASE", "PCX", "BTS"]  # Nasdaq tiers, NYSE, NYSE American, NYSE Arca, Cboe
CA_EXCHANGES = ["TOR", "VAN", "NEO", "CNQ"]                       # TSX, TSX Venture, Cboe Canada, CSE
# OTC / pink-sheet codes are deliberately excluded: they are not exchange listings.
MAX_STALE_DAYS = 5           # last price older than this before the purchase date = not trading

# Strategy parameters
TRADING_DAYS_PER_YEAR = 252
LOOKBACK_DAYS = 252          # one year: Client B volatility and the equal-weight comparison period
MOMENTUM_DAYS = 126          # about six months
SHORT_MOMENTUM_DAYS = 20     # about one month
TREND_DAYS = 50              # 50-day moving average trend filter
CATALYST_BONUS = 0.10        # score bonus for Client A stocks with earnings in the window and positive momentum

BENCHMARK_TICKERS = {"US": "^GSPC", "CA": "^GSPTSE"}  # S&P 500 and S&P/TSX Composite
FX_TICKER = "CAD=X"          # CAD per 1 USD
PAUSE_SECONDS = 0.2          # pause between per-ticker requests to avoid rate limits





## read the ticker file and collect company information

def read_tickers(path):
    """Read the ticker file (one ticker per row, no header) in order to return a clean, de-duplicated list."""
    raw = pd.read_csv(path, header=None) ## pd haven't touched the data
    tickers = (raw.iloc[:, 0].dropna().astype(str)
               .str.strip().str.upper())
    tickers = tickers[tickers != ""].drop_duplicates()
    return list(tickers)


def get_stock_info(tickers):
    """Looking up quote type, exchange, currency and sector for each ticker.
    Tickers that yfinance cannot find will result in empty values (they fail the checks later)."""
    rows = []
    for ticker in tickers:
        try:
            info = yf.Ticker(ticker).info
        except Exception:
            info = {}
        rows.append({
            "Ticker": ticker,
            "SecurityType": info.get("quoteType"),
            "Exchange": info.get("exchange"),
            "Currency": info.get("currency"),
            "Sector": info.get("sector"),
            "Name": info.get("longName") or info.get("shortName"),
        })
        time.sleep(PAUSE_SECONDS)
    return pd.DataFrame(rows).set_index("Ticker")


def market_of(exchange):
    """Return 'US', 'CA' or None depending on where the stock is listed."""
    if pd.isna(exchange):
        return None
    if exchange in US_EXCHANGES:
        return "US"
    if exchange in CA_EXCHANGES:
        return "CA"
    return None



## download price and volume history
#### Use auto_adjust=False and Close column


def download_history(tickers, start, end):
    """Download daily closes and volumes
       return into two DFs (dates x tickers)"""

    data = yf.download(tickers, start=start, end=end + pd.Timedelta(days=1),
                       auto_adjust=False, progress=False, group_by="column")

    closes = data["Close"]
    volumes = data["Volume"]
    # With one ticker yfinance can return a Series; make it a DataFrame either way
    if isinstance(closes, pd.Series):
        closes = closes.to_frame(tickers[0])
        volumes = volumes.to_frame(tickers[0])
    closes.index = pd.to_datetime(closes.index).tz_localize(None)
    volumes.index = pd.to_datetime(volumes.index).tz_localize(None)
    return closes, volumes 



def get_fx_rate(index, start, end):
    """CAD per USD, lined up with the stock dates (forward-filled over holidays)."""
    fx = yf.download(FX_TICKER, start=start, end=end + pd.Timedelta(days=1),
                     auto_adjust=False, progress=False)["Close"]
    if isinstance(fx, pd.DataFrame):
        fx = fx.iloc[:, 0]
    fx.index = pd.to_datetime(fx.index).tz_localize(None)
    return fx.reindex(fx.index.union(index)).ffill().reindex(index).rename("USDCAD")



def convert_to_cad(closes_local, currencies, fx):
    """Multiply USD-quoted columns by the daily exchange rate; CAD columns stay as they are."""
    prices_cad = closes_local.copy()
    for ticker in closes_local.columns:
        if currencies.get(ticker) == "USD":
            prices_cad[ticker] = closes_local[ticker] * fx
    return prices_cad




## Volume Rule
## A "trading day" is a day the stock actually has a volume record. Because TSX and US holidays differ, each stock is counted on its own calendar.
## The average is taken over all remaining days (not an average of monthly averages).


def average_volume(volume_series, start, end, min_days):
    """Average daily volume over [start, end], excluding months with fewer than min_days trading days."""
    v = volume_series.loc[start:end].dropna()
    if v.empty:
        return np.nan
    month = v.index.to_period("M")
    days_per_month = v.groupby(month).count()
    full_months = days_per_month[days_per_month >= min_days].index
    v = v[month.isin(full_months)]
    return v.mean() if not v.empty else np.nan


## Eligibility check + load_stock_data()
def check_eligibility(info, closes_local, volumes, as_of):
    """Add AvgVolume, Eligible and Reason columns. Every failed rule is listed in Reason."""
    result = info.copy()
    result["Market"] = result["Exchange"].apply(market_of)
    result["AvgVolume"] = np.nan
    result["Eligible"] = False
    result["Reason"] = ""

    for ticker in result.index:
        problems = []
        row = result.loc[ticker]

        if pd.isna(row["SecurityType"]):
            # yfinance found nothing: report one clear reason and skip the other checks
            result.loc[ticker, "Reason"] = "invalid ticker (no data found)"
            continue
        if row["SecurityType"] not in ALLOWED_QUOTE_TYPES:
            problems.append(f"not a company stock ({row['SecurityType']})")
        if pd.isna(row["Market"]):
            problems.append(f"not listed on a Canadian/US exchange ({row['Exchange']})")
        if row["Currency"] not in ALLOWED_CURRENCIES:
            problems.append(f"currency not CAD/USD ({row['Currency']})")

        if ticker in closes_local.columns:
            prices = closes_local[ticker].dropna()
            last_date = prices.index.max() if not prices.empty else None
            check_date = min(as_of, pd.Timestamp.today().normalize())
            if last_date is None or (check_date - last_date).days > MAX_STALE_DAYS:
                problems.append("no recent price (possibly delisted)")
            avg_vol = average_volume(volumes[ticker], VOLUME_START, VOLUME_END,
                                     MIN_TRADING_DAYS_IN_MONTH)
            result.loc[ticker, "AvgVolume"] = avg_vol
            if pd.isna(avg_vol) or avg_vol < MIN_AVG_VOLUME:
                problems.append(f"average volume below {MIN_AVG_VOLUME:,}")
        else:
            problems.append("no price data")

        result.loc[ticker, "Eligible"] = len(problems) == 0
        result.loc[ticker, "Reason"] = "; ".join(problems)
    return result


def load_stock_data(path, as_of=PURCHASE_DATE):
    """Read tickers, check eligibility and download history.
    Returns a dictionary with the screening table and all historical tables."""
    tickers = read_tickers(path)
    print(f"Read {len(tickers)} unique tickers from {path}")

    info = get_stock_info(tickers)
    found = [t for t in tickers if pd.notna(info.loc[t, "SecurityType"])]
    closes_local, volumes = download_history(found, HISTORY_START, as_of)

    screening = check_eligibility(info, closes_local, volumes, as_of)
    eligible = list(screening.index[screening["Eligible"]])

    # Keep only eligible stocks in the price tables
    closes_local = closes_local[eligible].loc[:as_of]
    volumes = volumes[eligible].loc[:as_of]

    fx = get_fx_rate(closes_local.index, HISTORY_START, as_of)
    prices_cad = convert_to_cad(closes_local, screening["Currency"].to_dict(), fx)

    print(f"{len(eligible)} eligible, {len(tickers) - len(eligible)} excluded")
    return {
        "screening": screening,
        "eligible": eligible,
        "closes_local": closes_local,
        "volumes": volumes,
        "fx": fx,
        "prices_cad": prices_cad,
    }


## benchmarks in CAD

## Each stock is compared with home markets

def get_benchmarks_cad(index, fx, start, end):
    """Return a DataFrame with columns 'US' and 'CA', both in CAD, lined up with the stock dates."""
    symbols = list(BENCHMARK_TICKERS.values())
    closes, _ = download_history(symbols, start, end)
    closes = closes.reindex(closes.index.union(index)).ffill().reindex(index)
    benchmark_cad = pd.DataFrame(index=index)
    benchmark_cad["US"] = closes[BENCHMARK_TICKERS["US"]] * fx   # S&P 500 is in USD
    benchmark_cad["CA"] = closes[BENCHMARK_TICKERS["CA"]]        # TSX is already in CAD
    return benchmark_cad





## Catalysts
def earnings_in_window(timestamps, start=HOLD_START, end=HOLD_END):
    """Return the first earnings timestamp inside the window, or NaT."""
    for ts in sorted(timestamps):
        ts = pd.Timestamp(ts)
        if ts.tzinfo is None:
            ts = ts.tz_localize("America/New_York")
        else:
            ts = ts.tz_convert("America/New_York")
        if ts.hour == 0 and ts.minute == 0:      # time unknown: compare dates only
            if start.date() < ts.date() <= end.date():
                return ts
        elif start < ts < end:
            return ts
    return pd.NaT


def get_earnings_timestamps(ticker):
    """Collect earnings timestamps from get_earnings_dates(), falling back to .calendar."""
    stock = yf.Ticker(ticker)
    timestamps = []
    try:
        dates = stock.get_earnings_dates(limit=8)
        if dates is not None and len(dates) > 0:
            timestamps += list(dates.index)
    except Exception:
        pass
    if not timestamps:
        try:
            calendar = stock.calendar or {}
            timestamps += list(calendar.get("Earnings Date", []))
        except Exception:
            pass
    return timestamps


def get_catalysts(tickers):
    """Return a DataFrame with EarningsDate and CatalystStatus for each ticker."""
    rows = []
    for ticker in tickers:
        timestamps = get_earnings_timestamps(ticker)
        hit = earnings_in_window(timestamps)
        if pd.notna(hit):
            status = "Event"
        elif not timestamps:
            status = "Unknown"
        # Only say "NoEvent" if we can see an upcoming date (otherwise the data may just be stale)
        elif max(pd.Timestamp(t).tz_localize(None) if pd.Timestamp(t).tzinfo is None
                 else pd.Timestamp(t).tz_convert(None) for t in timestamps) > HOLD_START.tz_convert(None):
            status = "NoEvent"
        else:
            status = "Unknown"
        rows.append({"Ticker": ticker, "EarningsDate": hit, "CatalystStatus": status})
        time.sleep(PAUSE_SECONDS)
    return pd.DataFrame(rows).set_index("Ticker")



## Volatility

def window_return(prices, days):
    """Return over the last `days` trading days (prices must have no gaps)."""
    if len(prices) <= days:
        return np.nan
    return prices.iloc[-1] / prices.iloc[-1 - days] - 1


def calculate_signals(prices_cad, volumes, benchmark_cad, markets, as_of=PURCHASE_DATE):
    """Return a DataFrame of signals (one row per ticker), calculated as of `as_of`."""
    prices_cad = prices_cad.loc[:as_of]
    volumes = volumes.loc[:as_of]
    benchmark_cad = benchmark_cad.loc[:as_of]

    # Benchmark 20-day returns, using the benchmark's own trading days
    bench_return20 = {m: window_return(benchmark_cad[m].dropna(), 20) for m in benchmark_cad.columns}

    rows = []
    for ticker in prices_cad.columns:
        prices = prices_cad[ticker].dropna()
        daily_returns = prices.pct_change().dropna()
        vol = volumes[ticker].dropna()

        vol20 = daily_returns.tail(20).std() if len(daily_returns) >= 20 else np.nan
        vol60 = daily_returns.tail(60).std() if len(daily_returns) >= 60 else np.nan
        return20 = window_return(prices, 20)
        ma20 = prices.tail(20).mean() if len(prices) >= 20 else np.nan
        avg_vol_60 = vol.tail(60).mean() if len(vol) >= 60 else np.nan

        rows.append({
            "Ticker": ticker,
            "Vol20": vol20,
            "Vol60": vol60,
            "VolRatio": vol20 / vol60 if vol60 else np.nan,
            "Return5": window_return(prices, 5),
            "Return20": return20,
            "PriceVsMA20": prices.iloc[-1] / ma20 - 1 if ma20 else np.nan,
            "RelativeMomentum": return20 - bench_return20.get(markets.get(ticker), np.nan),
            "RelativeVolume": vol.tail(5).mean() / avg_vol_60 if avg_vol_60 else np.nan,
        })
    signals = pd.DataFrame(rows).set_index("Ticker")

    # Percentile ranks (0 to 1) make the signals comparable for Zac's scoring
    for column in ["Vol20", "Return20", "RelativeMomentum", "RelativeVolume"]:
        signals[column + "Rank"] = signals[column].rank(pct=True)
    return signals



## running everything

data = load_stock_data(TICKER_FILE)
screening = data["screening"]

# Show what was excluded and why
exclusions = screening.loc[~screening["Eligible"], ["SecurityType", "Exchange", "Currency", "Reason"]]
print(f"\nExcluded {len(exclusions)} tickers:")
display(exclusions)

Read 40 unique tickers from /Users/kenjvcs/Desktop/Tickers.csv


$MON: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2024-10-01 00:00:00 -> 2026-11-28 00:00:00)
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found

5 Failed downloads:
['MON', 'AGN', 'RTN', 'CELG']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2024-10-01 00:00:00 -> 2026-11-28 00:00:00)


35 eligible, 5 excluded

Excluded 5 tickers:


,SecurityType,Exchange,Currency,Reason
Ticker,,,,
AGN,NONE,NaN,NaN,not a company stock (NONE); not listed on a Ca...
BK,EQUITY,NYQ,NaN,currency not CAD/USD (nan); no recent price (p...
CELG,NONE,NaN,NaN,not a company stock (NONE); not listed on a Ca...
MON,NONE,NaN,NaN,not a company stock (NONE); not listed on a Ca...
RTN,NONE,NaN,NaN,not a company stock (NONE); not listed on a Ca...


In [5]:
test = yf.Ticker("BNY")
print(test.history(period="5d"))
print(test.info.get("currency"), test.info.get("exchange"))

                                 Open        High         Low       Close  \
Date                                                                        
2026-11-16 00:00:00-05:00  152.839996  153.350006  150.850006  151.690002   
2026-11-17 00:00:00-05:00  151.070007  152.559998  148.770004  149.789993   
2026-11-18 00:00:00-05:00  149.570007  149.850006  147.699997  147.800003   
2026-11-19 00:00:00-05:00  149.559998  149.970001  144.410004  144.830002   
2026-11-20 00:00:00-05:00  149.559998  144.910004  142.253906  144.279999   

                            Volume  Dividends  Stock Splits  
Date                                                         
2026-11-16 00:00:00-05:00  2901500        0.0           0.0  
2026-11-17 00:00:00-05:00  3005700        0.0           0.0  
2026-11-18 00:00:00-05:00  3108600        0.0           0.0  
2026-11-19 00:00:00-05:00  2308100        0.0           0.0  
2026-11-20 00:00:00-05:00   444615        0.0           0.0  
USD NYQ
